<img src="../logo.png" alt="Vegeta" width="240">

# ÉMERILLON — an electric speed quadcopter, and what each km/h costs (37)

Émerillon is the merlin in French: the small falcon that catches its prey by outrunning it in level flight. The
machine is the layout the speed-record quadcopters converged on — a slender body that stands on its tail, lifts off as
a quadcopter, pitches over and flies nose-first with four propellers pulling along its axis. Luke and Mike Bell's
Peregreen V4 holds the record at **657.59 km/h** (Guinness World Records, Cape Town, 11 December 2025: the mean of two
runs in opposite directions; four T-Motor 3120 motors at 900 KV, a 3D-printed airframe).

**The question.** The design goal is speed, electric. *How much speed does a dollar buy, and where does it stop being
worth it?* So this notebook does not design one aircraft: it prices every part on the market, builds every
combination the catalogue allows (336 of them), computes each one's top speed, and reads the front between speed and
price. ÉMERILLON as drawn here is the knee of that front.

**What is new, what is reused.** The propeller solver is Boreas's BEMT with a blade section that knows its Mach number
(`ef.MachAirfoil`: at 500 km/h the tips are transonic, and Boreas alone has no compressibility); friction comes from
`og.aero`, the mass budget from `og.mass`. New: the priced catalogue (`emerillon_market.py`), the drive under its
current limits and the pack's sag, the trim of a body flying on its own lift, the trade (`emerillon_flight.py`) and
the CAD (`emerillon.py`).

**How far to trust it.** Every number here is a hand model or a shop listing — nothing was measured and no solver
ran in the committed notebook. The prices are search results of one day (see section 2). Section 3 says what the
experimental Kestrel studies taught about such models, and which margins that put into this one.

In [ ]:
# DEBUG — ON BY DEFAULT: the expensive solvers are mocked (vegeta.mock: fixed numbers, seconds instead of hours,
# labelled MOCK). For real results set DEBUG = False here, or from the shell: VEGETA_DEBUG=0 jupyter nbconvert …
DEBUG = True
from vegeta.labkit import bootstrap
DEBUG = bootstrap.lab("aero", DEBUG)        # the lab's folders on sys.path, the mocks

In [ ]:
import dataclasses, json, math, os, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vegeta import dedalus, talos, aeromant, mellonia, core
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.mellonia.examples import GENERIC_PLA_0_2MM
import emerillon as em
import emerillon_flight as ef
import emerillon_market as mk

ROOT = Path("../_runs/emerillon"); shutil.rmtree(ROOT, ignore_errors=True); ROOT.mkdir(parents=True)
OUT = Path("../output") / "37_emerillon"; OUT.mkdir(parents=True, exist_ok=True)
RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM") != "1"
RERUN_TRADE = os.environ.get("EMERILLON_TRADE") == "run" and not DEBUG      # a minute and a half; otherwise the stored trade
DESIGN, FAST = ef.DESIGN, ef.RECORD_BUILD
COLOR = {"budget": "#2ca02c", "knee": "#d62728", "fastest": "#1f77b4"}
pd.set_option("display.precision", 3); pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 90)
print(f"CFD {'on if OpenFOAM is found' if RUN_CFD else 'skipped'}; trade {'re-run' if RERUN_TRADE else 'read from designs/data/emerillon_trade.json'}")

In [ ]:
import yanus                                       # Ozaru's requirement checks: what this notebook needs on this machine
from yanus import Package
NEEDS = [Package("vegeta.aeromant"), Package("vegeta.boreas"), Package("vegeta.core"), Package("vegeta.dedalus"),
         Package("vegeta.mellonia"), Package("vegeta.talos")]
print(yanus.table(yanus.check_needs(NEEDS)) if NEEDS else "needs: nothing beyond the Python environment")

# Part 1 — the task and the market

## 1. The record and the requirements — edit here

A record pass is a timed 100 m in level flight, flown in both directions and averaged (the wind cancels). The
requirements below are what `ef.evaluate` checks of every build; a build that misses one is not on the front.

In [ ]:
display(pd.Series(ef.RECORD, name="the record to measure against").to_frame())
REQ = pd.DataFrame({
    "requirement": ["electric, four rotors, takes off and lands on its tail", f"static thrust ≥ {ef.STATIC_TW_MIN:.0f} × weight",
                    "two passes in opposite directions on one charge", f"winding temperature rise ≤ {ef.MOTOR_DT_MAX:.0f} K per pass",
                    "motor and ESC inside their voltage", f"cells not pulled under {ef.CELL_V_MIN} V", "propeller inside its maker's rpm limit"],
    "why": ["the record's class", "it has to hover, tip over and catch itself", "the record is the mean of two runs",
            "magnets and enamel; no cooling counted in a pass of seconds", "what the listings allow", "the pack survives the day",
            "reported, not required: the record flies beyond it on proof-spun propellers"],
    "where": ["layout", "ef.speed_run", f"ef.speed_run, {ef.USABLE:.0%} of the pack, {ef.HOVER_S:.0f} s of hover per pass", "ef.speed_run",
              "ef.candidates", "ef.drive", "ef.evaluate: within_rpm_limit"]})
display(REQ)

## 2. The market — what the parts cost

`emerillon_market.py` holds every bought part as one row: mass, the electrical numbers the flight model needs, the
price and the shop. **Read the `kind` column before the price:** `sourced` and `partly` rows have a price from a shop
listing (`partly`: some electrical numbers are assumptions, named in the note); `assumed` rows had no listing and are
estimates. The prices were collected through a web search on the date printed below — search snippets, with listings
out of stock and sale prices among them. `lowest seen` / `highest seen` is the spread between shops for the same part.

Two things the search could not find, and the trade lives with: a **3120 motor at 900 KV** (the record's motor — the
shops list the 3120 at 450–700 KV, the 900 KV winding only on the smaller 3115) and a **7-inch propeller with more
than 10 inches of pitch** (the `own…` rows: a blade the lab would make itself).

In [ ]:
print("priced on", mk.PRICED_ON, "— USD; euro prices at", mk.USD_PER_EUR, "USD/EUR (assumed)")
for title, parts in (("motors", mk.MOTORS), ("propellers", mk.PROPELLERS), ("ESCs", mk.ESCS), ("batteries (each a 6S pack)", mk.BATTERIES),
                     ("what every build carries", mk.FIXED), ("filament [USD/kg]", mk.FILAMENTS)):
    print("\n" + title)
    display(pd.DataFrame(mk.price_table(parts)).drop(columns="group").set_index("part"))

In [ ]:
T = pd.DataFrame(mk.price_table(mk.MOTORS + mk.PROPELLERS + mk.ESCS + mk.BATTERIES + mk.FIXED))
fig, ax = plt.subplots(figsize=(11, 6.5))
y = np.arange(len(T))[::-1]
for yi, (_, r) in zip(y, T.iterrows()):
    lo, hi = (r["lowest seen"], r["highest seen"]) if pd.notna(r["lowest seen"]) else (r["price [USD]"], r["price [USD]"])
    ax.plot([lo, hi], [yi, yi], color="0.6", lw=4, solid_capstyle="round")
    ax.plot(r["price [USD]"], yi, "o", color={"sourced": "#2ca02c", "partly": "#1f77b4", "assumed": "#d62728"}[r["kind"]])
ax.set_yticks(y); ax.set_yticklabels(T["part"], fontsize=8); ax.set_xscale("log")
ax.set(xlabel="price per piece [USD, log]", title="the market: the price used (dot) and the spread between shops (bar)")
for k, c in (("sourced", "#2ca02c"), ("partly sourced", "#1f77b4"), ("assumed", "#d62728")):
    ax.plot([], [], "o", color=c, label=k)
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="x"); fig.tight_layout()

## 3. What the experimental Kestrel designs teach

Kestrel is the lab's tail-sitter family (`labs/aero/exp/`: 40–49, `designs/kestrel3.md`, `K4Q_vs_X.md`,
`RAPTORS_REDESIGN.md`). It flew at 16 m/s and topped out at 30–43 m/s, so its numbers do not carry to 100–180 m/s —
it says nothing on tip Mach, compressibility or propellers at high advance ratio. Its lessons do carry:

| what Kestrel found | where | what ÉMERILLON does with it |
|---|---|---|
| "V max is the propeller's pitch speed at the motor's full rpm more than it is drag"; four airframes' top speeds within 3 m/s of each other | `exp/44`, `exp/41`, `kestrel3.md` § 3 | the trade varies motor KV, cells and **pitch** first; section 9 shows the design is voltage-limited at J ≈ pitch/diameter, and section 11 that the drag margin moves it little |
| the mini raptors' speed was "the 5x5's pitch speed and the pack's C rating" | `exp/49` | the pack's sag and a cell-voltage floor are in the drive (`ef.drive`); the fast builds end up battery-limited |
| the hand drag build-up ran low against RANS: CD 0.065 against 0.026 on 4Q-2 (2.5×), 0.071 against 0.067 on 4Q-1 | `exp/43` cell 7 | `ef.DRAG_MARGIN` = 1.6 on the build-up, a sweep of 1.0–2.5 in section 11, and the CFD cell in section 6 is what replaces it |
| round struts were 73 % of the drag, faired ones 23 % | `exp/43` cell 6 | the arms are strut sections, the motors sit in pods; pods and arms are still the largest drag items (section 6) |
| printed parts weighed about twice their budget | `exp/49` | `ef.PRINT_FACTOR` = 1.8 on the computed skins |
| a tail-sitter's clean airframe "cannot shed speed" | `exp/42` | the field length in section 10 counts the acceleration twice; braking is an open item |
| a poor propeller calibration was written to file anyway and moved the hover rpm by 16 % | `exp/38` | the propeller maps here are not calibrated at all and say so; a bench run of the chosen propeller comes before any order |
| the notebook never launches a long solver; every number is labelled analytic / empirical / solver | `exp/43` § 0 | the committed notebook is unexecuted, runs under the mocks by default, and the CFD and FEA cells print NOT RUN without their tools |

# Part 2 — the aircraft

## 4. The airframe (Dedalus)

`ef.geometry(config)` sizes the airframe from the parts: the body's diameter is the pack's diagonal plus skins, its
bay as long as the packs in line, the arms long enough for the propeller tips to clear the body and each other.
`em.Emerillon` builds those numbers: an ogive nose, the bay, a tail cone, four strut-section arms with motor pods (the
propeller planes ahead of the arms' leading edges), four fins between the propeller wakes that are also its feet.

In [ ]:
design_file = ROOT / "emerillon.py"
shutil.copy(Path("../designs/emerillon.py"), design_file)
edesign = dedalus.load_design(f"{design_file}:Emerillon")
display(pd.DataFrame(edesign.params.table()).set_index("name"))
GEOM = {"design": edesign.generate(**ef.cad_parameters(DESIGN)), "fast": edesign.generate(**ef.cad_parameters(FAST))}
for k, g in GEOM.items():
    c = DESIGN if k == "design" else FAST
    geo = ef.geometry(c)
    print(f"{k:<7s} valid {g.shape.isValid()}, {len(g.shape.Solids())} solid, box {np.round(g.dimensions, 0)} mm, wetted {g.surface_area / 1e6:.3f} m² "
          f"(the flight model's areas: {geo['s_body'] + geo['s_arms'] + geo['s_pods'] + geo['s_fins']:.3f} m²)")
L = em.Emerillon.layout(em.resolve(ef.cad_parameters(DESIGN)))
print({k: round(v, 1) for k, v in L.items()})

In [ ]:
try:
    dviz.show(dviz.plot3d(GEOM["design"]))
except Exception as e:
    print("3D view needs pyvista:", e)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2), gridspec_kw={"width_ratios": [2.2, 3.6, 1.3]})
for a, (name, c) in zip(ax[:2], (("ÉMERILLON (the knee build)", DESIGN), ("the fast end of the front", FAST))):
    o = em.outline(ef.cad_parameters(c))
    for poly in o["side"]:
        a.fill(poly[:, 0], poly[:, 1], color="#d62728", alpha=0.35, ec="k", lw=0.6)
    for off in o["prop_offsets"]:
        a.plot([o["prop_x"]] * 2, [off - o["prop_half"], off + o["prop_half"]], color="k", lw=1.5)
    a.set_aspect("equal"); a.set(xlabel="x [m]", title=f"{name}: {o['length'] * 1000:.0f} mm, {c.cells}S{c.n_parallel}P"); a.grid(alpha=0.3)
o = em.outline(ef.cad_parameters(DESIGN))
ax[2].fill(*o["front"]["body"].T, color="#d62728", alpha=0.35, ec="k")
for d_, p_ in zip(o["front"]["props"], o["front"]["pods"]):
    ax[2].plot(*d_.T, color="k", lw=0.8); ax[2].fill(*p_.T, color="#d62728", alpha=0.35, ec="k", lw=0.6)
ax[2].set_aspect("equal"); ax[2].set(title="front view: the propeller disks"); ax[2].grid(alpha=0.3)
fig.tight_layout()

## 5. Mass and the bill of materials

The bought parts weigh what their listings say (or an assumption, marked); the printed parts are the CAD's skin areas
at the wall thickness, times `PRINT_FACTOR`. The price is the bill of materials: four motors, two sets of propellers
(the first hard landing takes one), the ESC, the packs, the electronics every build carries, and the filament at 1.6×
the flying grams. Not in it: the printer, the radio, goggles, a charger, shipping, tax.

In [ ]:
BOM = pd.DataFrame(mk.bom(DESIGN)).set_index("part")
display(BOM.drop(columns="url"))
b = ef.mass_budget(DESIGN)
props = b.properties()
print(f"all-up mass {b.mass * 1e3:.0f} g  ({', '.join(f'{k} {v * 1e3:.0f} g' for k, v in b.by_group().items())})")
print(f"centre of gravity {props.com[0] * 1e3:.0f} mm from the nose tip; the propeller plane at {ef.geometry(DESIGN)['x_prop'] * 1e3:.0f} mm "
      f"(standing on its tail, the weight hangs below the rotors)")
print(f"price {mk.price(DESIGN):.0f} USD as listed — {mk.price(DESIGN, which='low'):.0f} at the cheapest listing of every part, "
      f"{mk.price(DESIGN, which='high'):.0f} at the dearest")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for a, col, unit in ((ax[0], "line [USD]", "USD"), (ax[1], "mass [g]", "g")):
    s = BOM.groupby("group")[col].sum().sort_values()
    a.barh(s.index, s.values, color="#d62728", alpha=0.7); a.set(xlabel=unit, title=f"{col.split(' ')[0]} by group: {s.sum():.0f} {unit}"); a.grid(alpha=0.3, axis="x")
fig.tight_layout()

## 6. Drag

`ef.drag_buildup`: turbulent flat-plate friction with a compressibility correction (`og.aero.cf_turbulent`), form
factors (Raymer for the bodies of revolution, Hoerner for the strut), the arms and pods at the propeller slipstream's
dynamic pressure, 25 % interference on the junctions, then the margin of section 3. The CFD cell replaces the total
where OpenFOAM runs (one RANS of the solid aircraft at the design's top speed, Mach 0.3; the fast build at Mach 0.5
needs the compressible template and is an open item).

In [ ]:
V_D = ef.top_speed(DESIGN)["V"]
DB = {name: ef.drag_buildup(c, v) for name, c, v in (("ÉMERILLON", DESIGN, V_D), ("fast build", FAST, ef.top_speed(FAST)["V"]))}
display(pd.DataFrame({k: {**{f"{p} [cm²]": d[p] * 1e4 for p in ("body", "arms", "pods", "fins", "base", "clean")},
                          "× margin": d["margin"], "CdA [cm²]": d["CdA"] * 1e4, "Cd on the frontal area": d["Cd_frontal"]} for k, d in DB.items()}).round(2))
fig, ax = plt.subplots(figsize=(8, 3))
left = 0.0
for p, col in zip(("body", "arms", "pods", "fins", "base"), ("#1f77b4", "#ff7f0e", "#d62728", "#2ca02c", "0.5")):
    v = DB["ÉMERILLON"][p] * 1e4
    ax.barh(["clean build-up"], [v], left=left, color=col, label=f"{p} {v:.1f}"); left += v
ax.barh(["with the margin"], [DB["ÉMERILLON"]["CdA"] * 1e4], color="k", alpha=0.5)
ax.set(xlabel="drag area CdA [cm²]", title=f"ÉMERILLON at {V_D * 3.6:.0f} km/h"); ax.legend(fontsize=8, ncol=5, loc="lower right"); ax.grid(alpha=0.3, axis="x")
fig.tight_layout()

In [ ]:
CFD = None
env = None
if RUN_CFD:
    try:
        env = aeromant.OpenFOAMEnvironment.detect()
    except Exception as e:
        print("no OpenFOAM here:", e)
if env is not None:
    ws_cfd = core.Workspace.create(ROOT / "cfd", name="EMERILLON CFD")
    d_cfd = ws_cfd.add_design("emerillon", f"{design_file}:Emerillon")
    rev = d_cfd.new_revision(**ef.cad_parameters(DESIGN), angle_of_attack_deg=0.0, note="the solid aircraft, axial flow")
    rev.generate(stl_tolerance=0.2)
    geo = ef.geometry(DESIGN)

    def case(rev, workdir):
        return aeromant.CFDCase("rans_ksst_external", rev.stl,
            dict(velocity=V_D, kinematic_viscosity=ef.NU, density=ef.RHO, reference_area=geo["frontal"], reference_length=geo["length"],
                 center_of_rotation=(geo["x_arm"], 0.0, 0.0), iterations=400, residual_target=1e-4, surface_level=4, near_level=3,
                 wake_level=2, cells_per_length=2.0), workdir=workdir, geometry_units="mm", environment=env)
    ev = rev.run_cfd("axial", case, progress=True)
    if ev.ok:
        CFD = ev.metrics["Cd"] * geo["frontal"]
        clean = ef.drag_buildup(DESIGN, V_D)["clean"]
        print(f"RANS{' (MOCK)' if DEBUG else ''}: CdA {CFD * 1e4:.1f} cm² without propellers — the build-up's clean {clean * 1e4:.1f} cm², "
              f"so the margin it supports is {CFD / clean:.2f} (assumed: {ef.DRAG_MARGIN})")
    else:
        print("CFD not run:", ev.messages)
if CFD is None:
    print(f"CFD: NOT RUN — the build-up with its assumed margin of {ef.DRAG_MARGIN} stands")

## 7. The propeller at speed

Boreas's BEMT has no compressibility. `ef.MachAirfoil` gives each blade station its own Mach number (rotation and
flight speed together): Prandtl–Glauert on the lift slope, Korn's drag-divergence Mach number for the station's
thickness and lift, Lock's fourth-power drag rise above the critical Mach number, capped where a thin section's wave
drag levels off. `ef.propeller_map` solves a propeller over advance ratio and rotational tip Mach once; everything
after that is a table lookup. The blades are generic (constant geometric pitch, a planform from three chord fractions):
the makers publish no blade tables for these, so **the maps are uncalibrated** — a bench run comes first.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for a, key in zip(ax[:2], (DESIGN.propeller, FAST.propeller)):
    pm = ef.propeller_map(key)
    J = np.array(pm["J"])
    for mt, ct, cp in zip(pm["Mt"], pm["CT"], pm["CP"]):
        ct, cp = np.array(ct), np.array(cp)
        eta = np.where((ct > 0) & (cp > 0), ct * J / np.maximum(cp, 1e-9), np.nan)
        a.plot(J, eta, label=f"tip Mach {mt:.2f} ({mt * ef.A_SOUND / (math.pi * pm['D']) * 60:,.0f} rpm)")
    a.set(xlabel="advance ratio J = V / (n D)", ylabel="efficiency", ylim=(0, 1), title=mk.part(key).name); a.grid(alpha=0.3)
ax[0].legend(fontsize=7, title="rotational tip Mach")
rpms = np.linspace(8000, 40000, 17)
Jd = 0.8 * mk.part(DESIGN.propeller).p_in / mk.part(DESIGN.propeller).d_in
D_ = ef.propeller_map(DESIGN.propeller)["D"]
for comp, ls in ((True, "-"), (False, "--")):
    pts = [ef.solve_point(DESIGN.propeller, r, Jd * r / 60 * D_, compressible=comp) for r in rpms]
    ax[2].plot(rpms, [p.efficiency for p in pts], ls, color="#d62728", label="with the Mach-aware section" if comp else "Boreas alone")
ax[2].axvline(mk.part(DESIGN.propeller).rpm_limit_k / mk.part(DESIGN.propeller).d_in, color="k", ls=":", label="the maker's rpm limit")
ax[2].set(xlabel="rpm", ylabel="efficiency", title=f"{mk.part(DESIGN.propeller).name} at J = {Jd:.2f}: what compressibility takes"); ax[2].legend(fontsize=8); ax[2].grid(alpha=0.3)
fig.tight_layout()

## 8. The drive: motor, ESC, pack

Four first-order motors (Kv, resistance, no-load current) on one bus. The bus sags with the pack's resistance (cells
in series, packs in parallel; 7.5 mΩ·Ah per cell at 100C, assumed). Full throttle unless a limit holds first: the
motor's burst current, the ESC's rating, or the pack at its cell-voltage floor — then the throttle that just meets it.

In [ ]:
speeds = np.linspace(2, 200, 34)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
DR = {}
for name, c, col in (("ÉMERILLON", DESIGN, COLOR["knee"]), ("fast build", FAST, COLOR["fastest"])):
    pts = [ef.balance(c, float(v)) for v in speeds]
    DR[name] = pts
    ax[0].plot(speeds * 3.6, [p["thrust"] for p in pts], color=col, label=f"{name}: thrust")
    ax[0].plot(speeds * 3.6, [p["need"] for p in pts], "--", color=col, label=f"{name}: needed")
    ax[1].plot(speeds * 3.6, [p["current"] for p in pts], color=col, label=f"{name}: per motor")
    ax[1].axhline(min(mk.part(c.motor).max_a, mk.part(c.esc).max_a), color=col, ls=":", lw=0.8)
    ax[2].plot(speeds * 3.6, [p["cell_v"] for p in pts], color=col, label=name)
ax[0].set(xlabel="km/h", ylabel="N", ylim=(0, None), title="thrust available and thrust needed in level flight"); ax[0].legend(fontsize=7); ax[0].grid(alpha=0.3)
ax[1].set(xlabel="km/h", ylabel="A", title="motor current (dotted: the tighter of motor and ESC limits)"); ax[1].legend(fontsize=7); ax[1].grid(alpha=0.3)
ax[2].axhline(ef.CELL_V_MIN, color="k", ls=":"); ax[2].set(xlabel="km/h", ylabel="V per cell", title="the pack under load (dotted: the floor)"); ax[2].legend(fontsize=7); ax[2].grid(alpha=0.3)
fig.tight_layout()

## 9. Top speed

Level flight at speed: the body flies at a few degrees of incidence and carries the weight on its own normal force and
its arms (`ef.trim`), the thrust pays the drag. The top speed is the last speed where the thrust still covers it.

In [ ]:
EV = {"ÉMERILLON": ef.evaluate(DESIGN), "fast build": ef.evaluate(FAST)}
rows = ["kmh", "price", "mass_g", "limit", "rpm", "rpm_limit", "overspeed", "tip_mach", "J", "prop_eff", "current", "pack_current", "cell_v",
        "electric_kw", "thrust", "alpha_deg", "static_tw", "t_accel", "d_accel", "field_length", "wh_pass", "wh_pack", "passes", "motor_dT",
        "feasible", "within_rpm_limit", "assumed_parts"]
display(pd.DataFrame({k: {r: (round(float(v[r]), 2) if isinstance(v[r], (float, np.floating)) else v[r]) for r in rows} for k, v in EV.items()}))
d = EV["ÉMERILLON"]
pitch_speed = d["rpm"] / 60 * mk.part(DESIGN.propeller).p_in * 0.0254
print(f"ÉMERILLON: {d['kmh']:.0f} km/h, limited by {d['limit']} — the propeller's pitch speed at that rpm is {pitch_speed * 3.6:.0f} km/h "
      f"(J = {d['J']:.2f} against pitch/diameter = {mk.part(DESIGN.propeller).p_in / mk.part(DESIGN.propeller).d_in:.2f}); the motors draw "
      f"{d['current']:.0f} A of their {mk.part(DESIGN.motor).max_a:.0f} A")
print(f"fast build: {EV['fast build']['kmh']:.0f} km/h against the record's {ef.RECORD['kmh']} — limited by {EV['fast build']['limit']}, tips at Mach "
      f"{EV['fast build']['tip_mach']:.2f}, the propeller at {EV['fast build']['overspeed']:.2f} × the mark its proof spin has to pass")

## 10. The run

From a hover to 98 % of the top speed in a straight line, a held stretch through the 100 m gates, the same again the
other way. The energy counts the acceleration, the hold and 50 s of hover per pass; the windings take their copper
loss with no cooling counted.

In [ ]:
RUN = {}
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for name, c, col in (("ÉMERILLON", DESIGN, COLOR["knee"]), ("fast build", FAST, COLOR["fastest"])):
    m = ef.mass_budget(c).mass
    vt = EV[name]["V"]
    vs = np.linspace(3.0, 0.98 * vt, 40)
    pts = [ef.balance(c, float(v), m) for v in vs]
    acc = np.array([max(p["thrust"] * math.cos(math.radians(p["alpha_deg"])) - p["drag"], 1e-3) / m for p in pts])
    t = np.concatenate([[0], np.cumsum(np.diff(vs) / (0.5 * (acc[1:] + acc[:-1])))])
    x = np.concatenate([[0], np.cumsum(np.diff(t) * 0.5 * (vs[1:] + vs[:-1]))])
    RUN[name] = ef.speed_run(c, vt, m, n=40)
    ax[0].plot(t, vs * 3.6, color=col, label=name); ax[1].plot(x, vs * 3.6, color=col, label=name)
ax[0].set(xlabel="s", ylabel="km/h", title="the acceleration"); ax[1].set(xlabel="m", ylabel="km/h", title="speed against distance")
for a in ax:
    a.legend(); a.grid(alpha=0.3)
fig.tight_layout()
display(pd.DataFrame(RUN).round(2))

# Part 3 — speed against price

## 11. The trade

`ef.candidates()` is every build the catalogue allows: motor × propeller × pack × (6S or 12S, one or two packs in
parallel), each with the cheapest ESC set that takes its voltage and its motor's burst current. `ef.trade` evaluates
them all (`labs/aero/sims/emerillon_trade.py` re-runs it and stores it; `EMERILLON_TRADE=run` re-runs it here). The
**front** is the set no other build beats on both price and speed. Three picks from it: the cheapest feasible build,
the **knee** (the point farthest above the straight line between the front's ends — past it each km/h costs more than
the front's average) and the fastest.

In [ ]:
ROWS = ef.trade() if RERUN_TRADE else ef.load_trade()
PK = ef.picks(ROWS)
PK_IN = ef.picks(ROWS, within_rpm_limit=True)
ok = [r for r in ROWS if r["feasible"]]
print(f"{len(ROWS)} builds, {len(ok)} feasible; what stops the others:",
      pd.Series([f for r in ROWS for f in r["fails"]]).value_counts().to_dict())
fig, ax = plt.subplots(figsize=(12, 6.5))
ax.scatter([r["price"] for r in ROWS if not r["feasible"]], [r["kmh"] for r in ROWS if not r["feasible"]], s=14, color="0.8", label="misses a requirement")
inside = [r for r in ok if r["within_rpm_limit"]]
over = [r for r in ok if not r["within_rpm_limit"]]
ax.scatter([r["price"] for r in inside], [r["kmh"] for r in inside], s=18, color="#2ca02c", alpha=0.6, label="feasible, propeller inside its maker's rpm limit")
ax.scatter([r["price"] for r in over], [r["kmh"] for r in over], s=18, color="#ff7f0e", alpha=0.6, label="feasible, propeller beyond it (proof spin needed)")
ax.step([r["price"] for r in PK["front"]], [r["kmh"] for r in PK["front"]], where="post", color="k", lw=1.2, label="the front")
ax.step([r["price"] for r in PK_IN["front"]], [r["kmh"] for r in PK_IN["front"]], where="post", color="#2ca02c", lw=1.2, ls="--", label="the front inside the rpm limits")
for k in ("budget", "knee", "fastest"):
    r = PK[k]
    ax.plot(r["price"], r["kmh"], "o", ms=12, mfc="none", mew=2.5, color=COLOR[k])
    ax.annotate(f"{k}: {r['kmh']:.0f} km/h, {r['price']:.0f} USD", (r["price"], r["kmh"]), textcoords="offset points", xytext=(10, -14), color=COLOR[k])
ax.axhline(ef.RECORD["kmh"], color="k", ls=":"); ax.text(ax.get_xlim()[0] + 10, ef.RECORD["kmh"] + 6, f"the record: {ef.RECORD['kmh']} km/h", fontsize=9)
ax.set(xlabel="price of the build [USD]", ylabel="top speed [km/h]", title="every build of the catalogue: speed against price"); ax.legend(fontsize=8, loc="center right"); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(OUT / "trade.png", dpi=130)

In [ ]:
cols = ["price", "kmh", "mass_g", "limit", "overspeed", "tip_mach", "prop_eff", "current", "passes", "motor_dT", "static_tw", "assumed_parts"]
FRONT = pd.DataFrame([{**{"build": r["label"]}, **{c: r[c] for c in cols}} for r in PK["front"]]).set_index("build")
FRONT["USD per km/h from the step before"] = [np.nan] + [m["usd_per_kmh"] for m in ef.marginal(PK["front"])]
FRONT["pick"] = ["/".join(k for k in ("budget", "knee", "fastest") if PK[k] is r) for r in PK["front"]]
display(FRONT.round(2))
print("inside the propeller makers' rpm limits:")
display(pd.DataFrame([{**{"build": r["label"]}, **{c: r[c] for c in cols}} for r in PK_IN["front"]]).set_index("build").round(2))
k = PK["knee"]
print(f"the knee: {k['label']} — {k['kmh']:.0f} km/h for {k['price']:.0f} USD ({k['price'] / k['kmh']:.2f} USD per km/h); "
      f"the fastest costs {PK['fastest']['price'] / k['price']:.1f} × as much for {PK['fastest']['kmh'] / k['kmh']:.2f} × the speed")
assert k["config"] == DESIGN or RERUN_TRADE, "the stored trade's knee is no longer the design: re-read section 11"

**Reading it.** The front has three steps.

1. **Around 400 USD: pitch.** On the cheapest motor, 6S and the freestyle stack, the speed is the propeller's pitch
   speed: changing a 3-dollar propeller from 6 to 7 to 10 inches of pitch is worth more than 100 km/h, for nothing.
   The motor is far from its current limit. This is Kestrel's lesson at five times the speed.
2. **600 to 1,100 USD: a plateau.** Bigger motors on the same shelf propellers buy little — the 7x10 is the most pitch
   a shop sells at 7 inches. The step that pays here is a propeller of the lab's own with more pitch.
3. **Above 1,100 USD: voltage and current.** 12S, one ESC per motor, four packs: the speed now comes from rpm, the
   tips go transonic (efficiency falls from about 0.8 to 0.6), the pack and the windings become the limits, and the
   last 75 km/h cost about as much as the first 380. Both parts that get there are not on a shelf: the 900 KV wind of
   the 3120 and the high-pitch propeller.

## 12. How much the assumptions move it

The drag margin and the printed-mass factor are the two guesses the hand model leans on; the price spread between
shops is the third uncertainty.

In [ ]:
margins = (1.0, 1.3, 1.6, 2.0, 2.5)
SENS = pd.DataFrame({name: {f"drag × {m}": ef.top_speed_with(c, drag_margin=m)["kmh"] for m in margins} |
                           {f"printed mass × {f}": ef.top_speed_with(c, print_factor=f)["kmh"] for f in (1.0, 1.8, 2.5)}
                     for name, c in (("ÉMERILLON [km/h]", DESIGN), ("fast build [km/h]", FAST))})
display(SENS.round(0))
PRICE = pd.DataFrame({name: {w: mk.price(c, which=w) for w in ("low", "used", "high")} for name, c in (("ÉMERILLON [USD]", DESIGN), ("fast build [USD]", FAST))})
display(PRICE.round(0))
print(f"a drag estimate 2.5 × low costs ÉMERILLON {SENS.iloc[0, 0] - SENS.iloc[4, 0]:.0f} km/h of {SENS.iloc[0, 0]:.0f}: it is pitch-limited, not drag-limited; "
      f"the weight hardly matters at all at top speed")

# Part 4 — structure and printing

## 13. The arm (Talos) and the prints (Mellonia)

The thrust acts along the body's axis, so it bends each arm in its own chord plane — the stiff way. The FEA holds the
arm at its root pad and pulls on the pod's flat motor face with the static full-throttle thrust of one motor. The
material is printed PA6-CF as a solid equivalent (assumed values: replace with coupon tests). Without CalculiX the cell
prints NOT RUN; under the mocks its numbers are labelled MOCK.

In [ ]:
PA6CF = talos.Material("printed PA6-CF (solid-equivalent)", youngs_modulus=3300.0, poissons_ratio=0.35, density=1.09e-9,
                       yield_strength=60.0, source="assumed: 60 % of the filament's datasheet values; replace with coupon tests")
ws = core.Workspace.create(ROOT / "workspace", name="EMERILLON")
design = ws.add_design("emerillon", f"{design_file}:Emerillon")
arm = design.new_revision(part="arm", **ef.cad_parameters(DESIGN), note="one arm for the FEA")
arm.generate()
pp = arm.params
LA = em.Emerillon.layout(pp)
F_MOTOR = RUN["ÉMERILLON"]["static_thrust"] / 4
r_pod = pp["pod_diameter"] / 2 + 1.0


def arm_model(rev):
    regions = [talos.SurfacesInBox("root", (LA["x_arm"] - 30.0, LA["R"] - 0.6, -40.0, LA["x_arm"] + pp["arm_chord"] + 30.0, LA["R"] - 0.4, 40.0)),
               talos.SurfacesInBox("mount", (LA["x_mount"] - 0.1, pp["arm_radius"] - r_pod, -r_pod, LA["x_mount"] + 0.1, pp["arm_radius"] + r_pod, r_pod))]
    return talos.StructuralModel(rev.step, "mm-N-MPa", PA6CF, regions, [talos.FixedSupport("root")],
                                 [talos.Force("mount", fx=-F_MOTOR)], talos.MeshSettings(element_size=3.0), name="static_thrust")


EV_ARM = arm.run_fea("static_thrust", arm_model, progress=False)
print(f"load: {F_MOTOR:.1f} N per motor (static, full throttle)")
if EV_ARM.ok:
    print({k: round(v, 3) if isinstance(v, float) else v for k, v in EV_ARM.metrics.items() if k in ("max_von_mises", "max_displacement", "safety_factor_yield")},
          "(MOCK)" if DEBUG else "")
else:
    print("arm FEA: NOT RUN —", EV_ARM.messages)

In [ ]:
nose = design.new_revision(part="nose", **ef.cad_parameters(DESIGN), note="nose cone, printed upright")
nose.generate()
PRINTS = {"arm (flat)": arm.run_print("flat", GENERIC_PLA_0_2MM, mellonia.Orientation()),
          "nose (upright)": nose.run_print("upright", GENERIC_PLA_0_2MM, mellonia.Orientation(rotate_y=-90))}
done = {n: {m: e.metrics.get(m) for m in ("estimated_time", "filament_used_g", "layer_count")} for n, e in PRINTS.items() if e.ok}
display(pd.DataFrame(done)) if done else print("slicing: NOT RUN —", next(iter(PRINTS.values())).messages)
print("(a generic PLA profile stands in: PETG and PA6-CF profiles are an open item)")

# Part 5 — ER1 and ER2: altitude, flights, and the solver chapters

## 14. Two variants

**ER1** is the cheapest build that flies the run (the front's first point, 261 km/h); **ER2** is the knee (383 km/h) —
the same motor, stack and airframe family, another propeller and pack. `ef.VARIANTS`, `ef.TARGET`.

In [ ]:
import emerillon_mission as mi
VAR = {}
for v, c in ef.VARIANTS.items():
    e = ef.evaluate(c)
    VAR[ef.VARIANT_NAME[v]] = {"build": c.label(), "price [USD]": e["price"], "top speed [km/h]": round(e["kmh"]), "target [km/h]": ef.TARGET[v]["kmh"],
                               "mass [g]": round(e["mass_g"]), "static thrust / weight": round(float(e["static_tw"]), 1), "propeller rpm / its limit": round(e["overspeed"], 2),
                               "passes on a pack": round(float(e["passes"]), 1), "windings ΔT per pass [K]": round(float(e["motor_dT"])),
                               "length [mm]": round(ef.geometry(c)["length"] * 1e3), "ceiling target [m]": ef.TARGET[v]["ceiling_m"]}
display(pd.DataFrame(VAR))

## 15. Altitude: 4 km for ER1, 7 km for ER2?

Thin air is not what stops them: at 7 km the static thrust is still several times the weight (the propellers turn
faster in thinner air, and the top speed even rises a little). **The pack is.** `mi.ceiling` climbs straight up in
slices, each at the throttle with the fewest watt-hours per metre, and asks whether the climb plus a landing fits in
the pack with a 20 % reserve. Lifting the mass costs m·g·h — 12 Wh for ER1 to 4 km, 22 Wh for ER2 to 7 km — and
the drive and the drag roughly double that. `mi.ceiling_options` tries every pack of the catalogue, one or two in
parallel. Not counted: a cold pack (−11 °C at 4 km, −30 °C at 7 km) gives less than a warm one, and nobody
may fly a model at those heights without the airspace's permission.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
CEIL = {}
for v, col in (("ER1", COLOR["budget"]), ("ER2", COLOR["knee"])):
    for cfg, ls, lab in ((v, "-", f"{v} as built"), (ef.HIGH[v], "--", f"{v} with {ef.HIGH[v].n_parallel} x {mk.part(ef.HIGH[v].battery).name}")):
        c = mi.ceiling(cfg, ef.TARGET[v]["ceiling_m"])
        CEIL[lab] = c
        pr = pd.DataFrame(c["profile"])
        ax[0].plot(pr["wh"], pr["h"], ls, color=col, label=lab)
        ax[0].plot([c["budget_wh"]] * 2, [0, c["energy_ceiling_m"]], ls=":", color=col, lw=0.8)
        ax[1].plot(pr["static_tw"], pr["h"], ls, color=col); ax[2].plot(pr["climb_ms"], pr["h"], ls, color=col)
    ax[0].axhline(ef.TARGET[v]["ceiling_m"], color=col, lw=0.6, alpha=0.6)
ax[0].set(xlabel="energy used in the climb [Wh] (dotted: what the pack allows with the reserve and a landing)", ylabel="altitude [m]", title="the climb's energy"); ax[0].legend(fontsize=7)
ax[1].set(xlabel="static thrust / weight", title="thrust is not the limit"); ax[2].set(xlabel="economy climb speed [m/s]", title="the climb")
for a in ax:
    a.grid(alpha=0.3)
fig.tight_layout()
display(pd.DataFrame({k: {m: (round(x, 1) if isinstance(x, float) else x) for m, x in c.items() if m in ("target_m", "climb_wh", "land_wh", "pack_wh", "budget_wh", "time_s", "energy_ceiling_m", "reaches_target", "static_tw_at_target", "T_C_at_target", "mass_g")} for k, c in CEIL.items()}))
for v in ef.VARIANTS:
    print(v, "— every pack of the catalogue:"); display(pd.DataFrame(mi.ceiling_options(v)).set_index("pack"))

**Reading it.** As built, neither reaches its target with the reserve: ER1 gets to about 3.5 km, ER2 to about
4.5 km. ER1 reaches 4 km on ER2's pack (one Tattu 1700 mAh, about 16 USD more, the same mass). ER2 needs two packs in
parallel for 7 km: with two Tattu 1700 mAh (about 49 USD and 310 g more) the slice analysis passes and the flown climb
of section 16 lands just under the reserve — 7 km is ER2's edge, not its comfort.

## 16. Six flights

`emerillon_mission.py` flies both variants as a point mass with a body axis: the tabulated drive along the axis, the
build-up's drag and the body's normal force, wind and terrain from the scenario, a guidance that asks for a velocity.
It gives **times, speeds, energy and margins, not handling** — no attitude dynamics, no propeller at an angle, no
vortex-ring state in the descents. The scenarios (`mi.SCENARIOS`; `labs/aero/sims/emerillon_mission.py` writes the
same figures to `sims/output/`):

- **range** — out against a 5 m/s wind to the radius of action and back at the best-range speed (section 16b; as built and with
  the long-range packs);
- **wildfire** — to a fire 2 km out, round the smoke column and back, at the fastest pace the pack allows;
- **mountain** — catch a target circling a 900 m peak at 137 km/h, come home (the base at 600 m above the sea);
- **sea** — catch a 90 km/h boat that zigzags, 12 m above it in a 14 m/s crosswind with gusts, hold over it, come home;
- **climb** — up to the ceiling target at the economy speed, a flat fall with the motors stopped, a catch and a landing
  (as built, and with the high-altitude pack);
- **palace** — a full-throttle climb to 1,200 m over Warsaw, a swoop, and a slow landing on the roof of the Palace of
  Culture and Science beside the spire, in a gusty city wind (roof 187.68 m and spire 237 m as commonly quoted: verify).

In [ ]:
FLIGHTS = mi.fly_all(scenarios=("palace", "mountain") if DEBUG else None)
display(pd.DataFrame(mi.table(FLIGHTS)).set_index(["scenario", "variant"]).sort_index())
for key in sorted(FLIGHTS, key=lambda k: (k[1], k[0])):
    if key[1] != "range" and (key[0] in ("ER2", "ER2 high") or key == ("ER1", "palace")):
        mi.plot(FLIGHTS[key], OUT / f"flight_{key[0].replace(' ', '_')}_{key[1]}.png")

## 16b. Long range

A bullet with four propellers is a sprinter. In level flight below about 100 km/h the body carries almost none of the
weight — the propellers do, at close to hover power — so the watt-hours per kilometre fall with speed until the drag
takes over: there is a **best-range speed**, far below the top speed (`mi.range_curve`; the range counts the pack less
the reserve and one take-off and landing). `mi.range_options` tries every pack; `ef.LONG` is the long-range fit (two
Tattu 1700 mAh in parallel). The `range` scenario flies it: out against a 5 m/s wind to 90 % of the predicted radius of
action and back, at the best-range airspeed.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
RANGE = {}
for v, col in (("ER1", COLOR["budget"]), ("ER2", COLOR["knee"])):
    for cfg, ls, lab in ((v, "-", f"{v} as built"), (ef.LONG[v], "--", f"{v} with 2 x {mk.part(ef.LONG[v].battery).name}")):
        r = mi.range_curve(cfg)
        RANGE[lab] = r
        rw = pd.DataFrame(r["rows"])
        ax[0].plot(rw["V"] * 3.6, rw["power_w"], ls, color=col, label=lab)
        ax[1].plot(rw["V"] * 3.6, rw["wh_per_km"], ls, color=col); ax[2].plot(rw["V"] * 3.6, rw["range_km"], ls, color=col)
        ax[2].plot(r["best_range_ms"] * 3.6, r["range_km"], "o", color=col)
ax[0].set(xlabel="km/h", ylabel="W", title="electrical power in level flight"); ax[0].legend(fontsize=7)
ax[1].set(xlabel="km/h", ylabel="Wh/km", ylim=(0, 10), title="energy per kilometre"); ax[2].set(xlabel="km/h", ylabel="km", title="range on the pack (dot: the best-range speed)")
for a in ax:
    a.grid(alpha=0.3)
fig.tight_layout()
keys = ("mass_g", "pack_wh", "usable_wh", "best_range_ms", "wh_per_km", "range_km", "radius_km", "range_time_s", "range_at_top_km", "best_endurance_ms", "endurance_s", "hover_endurance_s")
display(pd.DataFrame({k: {m: round(float(r[m]), 2) for m in keys} for k, r in RANGE.items()}))
for v in ef.VARIANTS:
    print(v, "— every pack of the catalogue:"); display(pd.DataFrame(mi.range_options(v)).set_index("pack"))
if ("ER2", "range") in FLIGHTS:
    for key in (("ER2", "range"), ("ER2 high", "range")):
        mi.plot(FLIGHTS[key], OUT / f"flight_{key[0].replace(' ', '_')}_range.png")

**Reading it.** As built ER1 goes about 8 km and ER2 about 11 km (a radius of 4 and 5.5 km) at about 140 km/h, five
minutes in the air; flat out, ER2's range is under 4 km. Two packs take both to about 18 km — the second pack's mass
costs a fifth of the energy per kilometre it brings. Nothing in the catalogue takes this airframe past 20 km: range is
what wings are for (NISUS, FALCO), and this machine has none.

## 17. The solver chapters — prepared, not run

The CFD and the FEA this notebook leaves NOT RUN are chapters of the workflow book **`emerillon`** (code **E**,
`labs/aero/campaigns/side_labs/`; `labs/aero/designs/emerillon_campaigns.py`), run like Kestrel's, on a machine with OpenFOAM and
CalculiX:

| chapter | what | cases |
|---|---|---|
| **E1** `airframe-drag` | the solid aircraft in axial flow at 50, 75, 100 % of the top speed → the drag area and the margin the hand build-up needs | 3 per variant |
| **E2** `airframe-alpha` | the body at 0–90° of incidence at 60 % of the top speed → the normal force of the trim and of the broadside braking | 8 per variant |
| **E3** `arm` | one printed arm: full-throttle thrust, a 5 g manoeuvre across it, a landing knock (ultimate ×1.5), and its modes against the rotor's 1P | 4 per variant |

```bash
source activate.sh
wf list                                             # the books; E1-E3 at the end
wf info E1 --variant ER2                            # the exact cases
wf E1 --variant ER2 --prepare-only                  # write the OpenFOAM cases, run nothing
wf E1 --variant ER2 --quality screening --detach    # run (10-20 min a case on 4 cores); wf runs / wf tail <run>
wf E2 --variant ER2 --detach
wf E3 --variant ER2                                 # minutes
```
The same for `--variant ER1`. The cell below lists the planned cases and reads the newest completed run of each
chapter when there is one. The CFD is incompressible RANS without propellers: right for ER1 (Mach 0.21) and ER2
(Mach 0.31), not for the fast end of the front.

In [ ]:
import sys
SIDE = Path("../campaigns/side_labs").resolve()
if str(SIDE) not in sys.path:
    sys.path.insert(0, str(SIDE))
import emerillon_campaigns as ec
import library, runkit
for v in ef.VARIANTS:
    print(ec.list_footer(ec.vehicle(v), "cfd"))
    for chapter, camp in (("airframe-drag", "drag"), ("airframe-alpha", "alpha")):
        run = library.latest_completed("emerillon", chapter, variant=v)
        summ = runkit.read_json(Path(run["run"]) / "summary.json") if run else None
        if summ and summ.get("records"):
            print(f"  {chapter}: run {Path(run['run']).name}", summ.get("finding") or "")
            display(pd.DataFrame(summ["records"]).set_index("case"))
        else:
            print(f"  {chapter}: NOT RUN — planned:"); display(ec.not_run_table(ec.cfd_specs(v, camp)))
    run = library.latest_completed("emerillon", "arm", variant=v)
    summ = runkit.read_json(Path(run["run"]) / "summary.json") if run else None
    print("  arm:", "run " + Path(run["run"]).name if summ else "NOT RUN — the beam rule meanwhile:")
    display(pd.DataFrame(summ["cases"]).T if summ else pd.Series({k: round(x, 2) for k, x in ec.arm_beam(v).items()}).to_frame(v))

# The verdict

- **ÉMERILLON as drawn** is the knee of the front: an EMAX ECO II 2807 at 1300 KV, an APC 7x10, the SpeedyBee F405
  stack, one Tattu R-Line 6S 1700 mAh — about **400 USD** of parts for about **380 km/h** by this model, every part on
  a shelf, the propeller inside its maker's rpm limit. Its margin is thin on one count: the windings' temperature rise
  over a pass is close to the limit.
- **The cheapest speed is pitch.** Below it, more money buys almost nothing until the propeller changes.
- **The record end** (about 640 km/h by this model against the record's 657.59) costs about 1,340 USD in parts and
  needs two things no shop sells: a 3120 wound to 900 KV and a 7-inch propeller with about 13 inches of pitch, spun
  beyond the mark a maker would set, with transonic tips. The model landing near the record with the record's motor is
  a sanity check of the model, not a prediction: the propeller maps are uncalibrated and the drag margin is a guess.
- **ER1 and ER2** are the front's first point and its knee. Both fly the wildfire, mountain, sea and palace scenarios
  with the reserve; the roof landing touches down under 1 m/s. **The altitude targets are a matter of the pack:** as
  built ER1 reaches about 3.5 km and ER2 about 4.5 km; ER1 makes 4 km on ER2's pack, ER2 makes 7 km on two packs and
  lands a little under the reserve.
- **Long range is not this machine's job:** about 8 km (ER1) and 11 km (ER2) as built at about 140 km/h, about 18 km on
  two packs; flat out ER2 goes under 4 km.
- **The arm's first bending mode (about 100–150 Hz by the beam rule) lies below the rotor's once-per-revolution
  frequency at top speed (over 400 Hz):** the rotors run through it on the way up. Chapter E3's modes say where.
- **What to do next, in order:** a bench run of the chosen motor and propeller (static thrust, current, rpm) to
  calibrate the map; the CFD chapter for the drag margin; the lab's own high-pitch blade from `prop_lab` with a proof
  spin; the pitch-over and the braking as a flight simulation; real quotes for the parts marked `assumed`.

In [ ]:
EXPORT = {"design": dataclasses.asdict(DESIGN), "label": DESIGN.label(), "priced_on": mk.PRICED_ON,
          "evaluation": {k: (float(v) if isinstance(v, (float, np.floating)) else v) for k, v in EV["ÉMERILLON"].items() if k != "config"},
          "cad_parameters": ef.cad_parameters(DESIGN), "bom": mk.bom(DESIGN), "drag_margin": ef.DRAG_MARGIN, "print_factor": ef.PRINT_FACTOR,
          "cfd_cda_m2": CFD, "mock": bool(DEBUG), "variants": {v: c.label() for v, c in ef.VARIANTS.items()},
          "ceilings": {k: {m: x for m, x in c.items() if m != "profile"} for k, c in CEIL.items()}, "flights": mi.table(FLIGHTS),
          "ranges": {k: {m: float(x) if isinstance(x, (float, np.floating)) else x for m, x in r.items() if m != "rows"} for k, r in RANGE.items()}}
(OUT / "emerillon_design.json").write_text(json.dumps(EXPORT, indent=1, default=str))
print("written:", OUT / "emerillon_design.json")